In [ ]:
# ==============================================================================
# TEST: merge SInAS_3.1.1 with variables_SInAS_manuela.csv + SInAS_locations_3.1.gpkg
#
# Run as a script or paste each "# %%" block into a notebook cell.
# Expects the three files in <repo_root>/data (repo_root = cwd.parent, as in your
# other notebooks). Change DATA below if needed.
#
# Goals
#   1. Merge location-level variables onto every SInAS row (LEFT join: no SInAS
#      row is ever dropped; missing values stay blank/NaN).
#   2. Link the polygons (gpkg) to the same location table.
#   3. Report how many SInAS regions have some data in the manuela file.
#   4. Show which API-derived columns (lat/lon, ISO3) are already available,
#      so the geocoding / ISO-mapping steps can be skipped in the pipeline.
# ==============================================================================

# %% 0. Setup
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import geopandas as gpd          # pip install geopandas pyogrio
except ImportError:
    gpd = None
    print("geopandas not installed: gpkg section will be skipped (pip install geopandas pyogrio)")

DATA        = Path.cwd().parent / "data"
SINAS_CSV   = DATA / "SInAS_3.1.1.csv"
MANUELA_CSV = DATA / "variables_SInAS_manuela.csv"
GPKG        = DATA / "SInAS_locations_3.1.gpkg"
SAVE        = False                  # True -> write the outputs listed at the bottom

# Columns of the manuela file that identify a region rather than describe it.
# Everything else counts as "data" when asking "does this region have any data?"
KEY_COLS = ["locationID", "location", "ISO3", "sinas_region"]
# Columns that legitimately vary per taxon group (not per location)
TAXON_LEVEL_COLS = ["taxaGroup", "native_richness"]

for p in (SINAS_CSV, MANUELA_CSV):
    assert p.exists(), f"Missing file: {p}"

In [ ]:
# %% 1. Load
def read_flexible(path, **kw):
    """The SInAS csv is space-separated with quotes in your sample; sniff the separator."""
    return pd.read_csv(path, sep=None, engine="python", quotechar='"', **kw)

sinas = read_flexible(SINAS_CSV)
sinas["locationID"] = pd.to_numeric(sinas["locationID"], errors="coerce").astype("Int64")
print(f"SInAS rows: {len(sinas):,} | columns: {list(sinas.columns)}")

man = read_flexible(MANUELA_CSV)
man = man.drop(columns=[c for c in man.columns if c.startswith("Unnamed")], errors="ignore")
man = man.replace(r"^\s*$", np.nan, regex=True)          # empty strings -> NaN
man["locationID"] = pd.to_numeric(man["locationID"], errors="coerce").astype("Int64")
print(f"manuela rows: {len(man):,} | columns: {list(man.columns)}")

# Several columns are typed as String in the csv (decimal commas?). Coerce numerics.
NUMERIC = ["latitude", "longitude", "temp_season", "precip_season",
           "gdp_capita_mean", "pop_dens_mean", "area_sqm",
           "island", "coastal", "native_richness", "years_colonial_activity"]
for c in NUMERIC:
    if c in man.columns:
        before = man[c].notna().sum()
        man[c] = pd.to_numeric(man[c].astype(str).str.replace(",", ".", regex=False),
                               errors="coerce")
        lost = before - man[c].notna().sum()
        if lost:
            print(f"   note: {c}: {lost} non-null values were not numeric and became NaN")


# %% 2. Structure of the manuela file: is it one row per location, or per location x taxaGroup?
n_loc_man = man["locationID"].nunique()
print(f"\nmanuela unique locationIDs: {n_loc_man} for {len(man)} rows "
      f"({len(man) / max(n_loc_man, 1):.1f} rows per location)")
if "taxaGroup" in man.columns:
    print("taxaGroup values:", man["taxaGroup"].value_counts(dropna=False).to_dict())

# Which columns are constant within a location (-> location-level) and which vary?
data_cols_all = [c for c in man.columns if c not in KEY_COLS + ["taxaGroup"]]
varies = {}
for c in data_cols_all:
    nun = man.groupby("locationID")[c].nunique(dropna=True)
    varies[c] = int((nun > 1).sum())
print("\nLocations with >1 distinct value per column (0 = safely location-level):")
for c, v in varies.items():
    print(f"   {c:26s} {v}")

STATIC_COLS = [c for c in data_cols_all
               if c not in TAXON_LEVEL_COLS]                 # to collapse per location
unexpected = [c for c in STATIC_COLS if varies[c] > 0]
if unexpected:
    print(f"\n WARNING: expected-static columns that vary within a location: {unexpected}. "
          "Taking the first non-null value; inspect these.")


# %% 3. Collapse manuela to ONE row per locationID
key_extra = [c for c in ["ISO3", "sinas_region"] if c in man.columns]
loc_tbl = (man.groupby("locationID")[key_extra + STATIC_COLS].first()      # first non-null
              .reset_index())

# name kept separately, only to cross-check against the SInAS location name
if "location" in man.columns:
    names_man = man.dropna(subset=["location"]).drop_duplicates("locationID")[["locationID", "location"]]
    loc_tbl = loc_tbl.merge(names_man.rename(columns={"location": "location_manuela"}),
                            on="locationID", how="left")

# taxon-level native richness -> wide columns native_richness_<taxaGroup>
if {"taxaGroup", "native_richness"} <= set(man.columns):
    rich = (man.pivot_table(index="locationID", columns="taxaGroup",
                            values="native_richness", aggfunc="max")
               .add_prefix("native_richness_").reset_index())
    loc_tbl = loc_tbl.merge(rich, on="locationID", how="left")

assert loc_tbl["locationID"].is_unique, "location table must be 1 row per locationID"
print(f"\nLocation-level table: {loc_tbl.shape[0]} rows x {loc_tbl.shape[1]} columns")


# %% 4. Compare region sets: SInAS vs manuela
sinas_regions = sinas[["locationID", "location"]].dropna(subset=["locationID"]).drop_duplicates("locationID")
sinas_ids, man_ids = set(sinas_regions["locationID"]), set(loc_tbl["locationID"].dropna())

print("\n=== REGION OVERLAP ===")
print(f"SInAS regions (locationID)            : {len(sinas_ids)}")
print(f"manuela regions                       : {len(man_ids)}")
print(f"in both                               : {len(sinas_ids & man_ids)}")
print(f"in SInAS only (no manuela row)        : {len(sinas_ids - man_ids)}")
print(f"in manuela only (not in this SInAS)   : {len(man_ids - sinas_ids)}")

# Name agreement for shared IDs (catches ID/name mismatches between releases)
chk = sinas_regions.merge(loc_tbl[["locationID"] + (["location_manuela"] if "location_manuela" in loc_tbl else [])],
                          on="locationID", how="inner")
if "location_manuela" in chk:
    bad = chk[chk["location"].str.strip().str.lower() != chk["location_manuela"].str.strip().str.lower()]
    print(f"shared IDs whose names differ         : {len(bad)}")
    if len(bad):
        print(bad.head(15).to_string(index=False))


# %% 5. MERGE (left join: every SInAS row kept, gaps left blank)
loc_for_merge = loc_tbl.drop(columns=["location_manuela"], errors="ignore")
clash = [c for c in loc_for_merge.columns if c in sinas.columns and c != "locationID"]
if clash:
    print(f"\n columns present in both files, manuela copy gets suffix '_man': {clash}")

merged = sinas.merge(loc_for_merge, on="locationID", how="left",
                     suffixes=("", "_man"), validate="m:1")

# ---- tests ----
assert len(merged) == len(sinas), "merge changed the number of SInAS rows"
assert merged["locationID"].equals(sinas["locationID"]), "row order/IDs changed"
print(f"\nMerged table: {merged.shape[0]:,} rows x {merged.shape[1]} columns (rows unchanged: OK)")


# %% 6. HOW MANY SInAS REGIONS HAVE SOME DATA IN THE MANUELA FILE?
DATA_COLS = [c for c in loc_tbl.columns
             if c not in KEY_COLS + ["location_manuela", "locationID"]]

reg = sinas_regions.merge(loc_tbl, on="locationID", how="left", suffixes=("", "_m"))
reg["in_manuela_file"] = reg["locationID"].isin(man_ids)
reg["n_vars_filled"]   = reg[DATA_COLS].notna().sum(axis=1)
reg["any_data"]        = reg["n_vars_filled"] > 0
reg["complete"]        = reg["n_vars_filled"] == len(DATA_COLS)

n = len(reg)
print("\n=== MANUELA COVERAGE OF SInAS REGIONS ===")
print(f"SInAS regions in total                          : {n}")
print(f"  present in the manuela file                   : {int(reg.in_manuela_file.sum())}")
print(f"  with >=1 data variable filled (any_data)      : {int(reg.any_data.sum())}  "
      f"({reg.any_data.mean():.1%})")
print(f"  with ALL {len(DATA_COLS)} data variables filled           : {int(reg.complete.sum())}  "
      f"({reg.complete.mean():.1%})")
print(f"  with NO data at all                           : {int((~reg.any_data).sum())}")

print("\nPer-variable coverage over SInAS regions:")
cov = reg[DATA_COLS].notna().sum().sort_values(ascending=False)
for c, v in cov.items():
    print(f"   {c:34s} {v:4d} / {n}  ({v / n:.1%})")

print("\nRegions with no data (first 40):")
print(reg.loc[~reg.any_data, ["locationID", "location"]].head(40).to_string(index=False))

# Row-weighted view: share of SInAS RECORDS that get at least one manuela variable
rows_any = merged[DATA_COLS].notna().any(axis=1).mean()
print(f"\nShare of SInAS records (rows) with >=1 manuela variable: {rows_any:.1%}")


# %% 7. Link the polygons (gpkg)
locations_geo = None
if gpd is None or not GPKG.exists():
    print("\n[gpkg] skipped:", "geopandas missing" if gpd is None else f"{GPKG} not found")
else:
    try:
        layers = gpd.list_layers(GPKG)
        print("\n[gpkg] layers:\n", layers)
    except Exception:
        pass
    gdf = gpd.read_file(GPKG)
    print(f"[gpkg] {len(gdf)} features | CRS: {gdf.crs} | columns: {list(gdf.columns)}")

    # Auto-detect the ID column: the one whose values overlap most with SInAS locationIDs
    best, best_n = None, -1
    for c in gdf.columns.drop("geometry"):
        vals = pd.to_numeric(gdf[c], errors="coerce").dropna().astype(int)
        ov = len(set(vals) & {int(i) for i in sinas_ids})
        if ov > best_n:
            best, best_n = c, ov
    print(f"[gpkg] ID column guess: '{best}' (matches {best_n} of {len(sinas_ids)} SInAS locationIDs)")
    ID_GPKG = best      # <- override by hand if the guess is wrong

    gdf["locationID"] = pd.to_numeric(gdf[ID_GPKG], errors="coerce").astype("Int64")
    # one geometry per location (dissolve multi-row locations)
    if not gdf["locationID"].is_unique:
        print("[gpkg] duplicated locationIDs -> dissolving to one geometry each")
        gdf = gdf.dissolve(by="locationID", as_index=False)

    geo_ids = set(gdf["locationID"].dropna())
    print(f"[gpkg] SInAS regions with polygon   : {len(sinas_ids & geo_ids)} / {len(sinas_ids)}")
    print(f"[gpkg] SInAS regions WITHOUT polygon: {len(sinas_ids - geo_ids)}")
    print(f"[gpkg] polygons not in this SInAS   : {len(geo_ids - sinas_ids)}")

    # Attach polygon to the location table (region-level, one row per location)
    locations_geo = (gdf[["locationID", "geometry"]]
                     .merge(reg.drop(columns=["in_manuela_file"]), on="locationID", how="right"))
    locations_geo = gpd.GeoDataFrame(locations_geo, geometry="geometry", crs=gdf.crs)

    # Centroids from polygons (equal-area CRS for a sane centroid, back to lat/lon)
    cent = (locations_geo.geometry.to_crs("EPSG:6933").centroid.to_crs("EPSG:4326"))
    locations_geo["centroid_lon"] = cent.x
    locations_geo["centroid_lat"] = cent.y

    # How much geocoding / API work could be skipped?
    if {"latitude", "longitude"} <= set(locations_geo.columns):
        have_ll   = locations_geo[["latitude", "longitude"]].notna().all(axis=1)
        have_poly = locations_geo.geometry.notna() & ~locations_geo.geometry.is_empty
        print("\n=== COORDINATE SOURCES (replaces Nominatim geocoding) ===")
        print(f"regions with lat/lon in manuela file          : {int(have_ll.sum())}")
        print(f"regions lacking lat/lon but WITH polygon      : {int((~have_ll & have_poly).sum())}  <- centroid from gpkg")
        print(f"regions lacking both (would need geocoding)   : {int((~have_ll & ~have_poly).sum())}")
        # sanity: manuela lat/lon vs polygon centroid
        d = locations_geo[have_ll & have_poly]
        err = np.hypot(d["latitude"] - d["centroid_lat"], d["longitude"] - d["centroid_lon"])
        print(f"manuela coord vs polygon centroid, median offset: {err.median():.2f} deg "
              f"| >10 deg off: {int((err > 10).sum())}  (check these, possible swapped/mismatched IDs)")

    if "ISO3" in locations_geo.columns:
        print(f"\nregions with ISO3 in manuela file             : {int(locations_geo['ISO3'].notna().sum())} / {len(locations_geo)}"
              "  (replaces SUB_NATIONAL_TO_ISO + country_converter)")


# %% 8. Optional: save outputs
if SAVE:
    out_dir = DATA
    loc_tbl.to_csv(out_dir / "SInAS_location_variables.csv", index=False)
    merged.to_csv(out_dir / "SInAS_merged_manuela.csv", index=False)
    if locations_geo is not None:
        locations_geo.to_file(out_dir / "SInAS_locations_enriched.gpkg", driver="GPKG")
    print("Saved: SInAS_location_variables.csv, SInAS_merged_manuela.csv"
          + (", SInAS_locations_enriched.gpkg" if locations_geo is not None else ""))